In [ ]:
# Import python packages
import streamlit as st
import pandas as pd

# We can also use Snowpark for our analyses!
from snowflake.snowpark.context import get_active_session
session = get_active_session()


In [ ]:
-- Créer la base, le schéma et le warehouse
CREATE DATABASE IF NOT EXISTS HOUSE_PRICE_DB;
CREATE SCHEMA IF NOT EXISTS HOUSE_PRICE_DB.ML;
CREATE WAREHOUSE IF NOT EXISTS ML_WH
  WAREHOUSE_SIZE = 'MEDIUM'
  AUTO_SUSPEND = 120
  AUTO_RESUME = TRUE;

USE DATABASE HOUSE_PRICE_DB;
USE SCHEMA ML;
USE WAREHOUSE ML_WH;

In [ ]:
import snowflake.snowpark as snowpark
from snowflake.snowpark.session import Session
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

print("Toutes les dépendances sont disponibles ✓")

In [ ]:
-- Créer le stage externe S3
CREATE OR REPLACE STAGE HOUSE_PRICE_STAGE
  URL = 's3://logbrain-datalake/datasets/house_price/'
  FILE_FORMAT = (TYPE = 'CSV' FIELD_OPTIONALLY_ENCLOSED_BY = '"' SKIP_HEADER = 1);

-- Créer la table cible
CREATE OR REPLACE TABLE HOUSE_PRICES (
    PRICE        NUMBER,
    AREA         NUMBER,
    BEDROOMS     NUMBER,
    BATHROOMS    NUMBER,
    STORIES      NUMBER,
    MAINROAD     VARCHAR(3),
    GUESTROOM    VARCHAR(3),
    BASEMENT     VARCHAR(3),
    HOTWATERHEATING VARCHAR(3),
    AIRCONDITIONING VARCHAR(3),
    PARKING      NUMBER,
    PREFAREA     VARCHAR(3),
    FURNISHINGSTATUS VARCHAR(20)
);

-- Charger depuis S3
COPY INTO HOUSE_PRICES
FROM @HOUSE_PRICE_STAGE
ON_ERROR = 'CONTINUE';

SELECT COUNT(*) FROM HOUSE_PRICES;

In [ ]:
-- Lister les fichiers dans le stage
LIST @HOUSE_PRICE_STAGE;

In [ ]:
CREATE OR REPLACE STAGE HOUSE_PRICE_STAGE
    URL = 's3://logbrain-datalake/datasets/house_price/'
    FILE_FORMAT = (
        TYPE = 'JSON'
        STRIP_OUTER_ARRAY = TRUE
    );

In [ ]:
-- Voir à quoi ressemble le fichier
SELECT $1
FROM @HOUSE_PRICE_STAGE
LIMIT 3;

In [ ]:
TRUNCATE TABLE HOUSE_PRICES;

COPY INTO HOUSE_PRICES (
    PRICE, AREA, BEDROOMS, BATHROOMS, STORIES,
    MAINROAD, GUESTROOM, BASEMENT, HOTWATERHEATING,
    AIRCONDITIONING, PARKING, PREFAREA, FURNISHINGSTATUS
)
FROM (
    SELECT 
        $1:price::NUMBER,
        $1:area::NUMBER,
        $1:bedrooms::NUMBER,
        $1:bathrooms::NUMBER,
        $1:stories::NUMBER,
        $1:mainroad::VARCHAR,
        $1:guestroom::VARCHAR,
        $1:basement::VARCHAR,
        $1:hotwaterheating::VARCHAR,
        $1:airconditioning::VARCHAR,
        $1:parking::NUMBER,
        $1:prefarea::VARCHAR,
        $1:furnishingstatus::VARCHAR
    FROM @HOUSE_PRICE_STAGE
)
FILE_FORMAT = (TYPE = 'JSON' STRIP_OUTER_ARRAY = TRUE)
ON_ERROR = 'CONTINUE';

-- Vérifier
SELECT COUNT(*) FROM HOUSE_PRICES;

-- Aperçu des données chargées
SELECT * FROM HOUSE_PRICES LIMIT 5;

In [ ]:
SELECT COUNT(*) FROM HOUSE_PRICES;

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

session = get_active_session()

# Charger la table
df = session.table("HOUSE_PRICES").to_pandas()

print(f"Dimensions : {df.shape}")
print(f"\nTypes :\n{df.dtypes}")
print(f"\nValeurs manquantes :\n{df.isnull().sum()}")

In [ ]:
df.describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['PRICE'], bins=30, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution des prix')

axes[1].hist(np.log(df['PRICE']), bins=30, color='teal', edgecolor='white')
axes[1].set_title('Distribution log(prix)')

plt.tight_layout()
plt.show()

In [ ]:
numeric_cols = ['PRICE', 'AREA', 'BEDROOMS', 'BATHROOMS', 'STORIES', 'PARKING']

plt.figure(figsize=(8, 6))
sns.heatmap(
    df[numeric_cols].corr(),
    annot=True,
    fmt='.2f',
    cmap='coolwarm',
    center=0,
    square=True
)
plt.title('Matrice de corrélation')
plt.show()

In [ ]:
cat_cols = ['MAINROAD', 'GUESTROOM', 'BASEMENT',
            'AIRCONDITIONING', 'PREFAREA', 'FURNISHINGSTATUS']

fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for i, col in enumerate(cat_cols):
    df.groupby(col)['PRICE'].mean().plot(
        kind='bar',
        ax=axes[i//3, i%3],
        color='steelblue',
        edgecolor='white'
    )
    axes[i//3, i%3].set_title(f'Prix moyen — {col}')
    axes[i//3, i%3].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Copie de travail
df_ml = df.copy()

# Encodage binaire yes/no → 1/0
binary_cols = ['MAINROAD', 'GUESTROOM', 'BASEMENT',
               'HOTWATERHEATING', 'AIRCONDITIONING', 'PREFAREA']
for col in binary_cols:
    df_ml[col] = df_ml[col].map({'yes': 1, 'no': 0})

# Encodage ordinal ameublement
df_ml['FURNISHINGSTATUS'] = df_ml['FURNISHINGSTATUS'].map({
    'furnished': 2,
    'semi-furnished': 1,
    'unfurnished': 0
})

# Features et cible
feature_cols = ['AREA', 'BEDROOMS', 'BATHROOMS', 'STORIES',
                'MAINROAD', 'GUESTROOM', 'BASEMENT', 'HOTWATERHEATING',
                'AIRCONDITIONING', 'PARKING', 'PREFAREA', 'FURNISHINGSTATUS']

X = df_ml[feature_cols]
y = df_ml['PRICE']

# Normalisation
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print(f"X_train : {X_train.shape}")
print(f"X_test  : {X_test.shape}")
print(f"\nAperçu des features encodées :")
print(pd.DataFrame(X_scaled, columns=feature_cols).head())

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb

# Définir les modèles
models = {
    'Linear Regression':   LinearRegression(),
    'Random Forest':       RandomForestRegressor(n_estimators=100, random_state=42),
    'Gradient Boosting':   GradientBoostingRegressor(n_estimators=100, random_state=42),
    'XGBoost':             xgb.XGBRegressor(n_estimators=100, random_state=42, verbosity=0)
}

# Entraîner et évaluer chaque modèle
results = {}
trained_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)
    
    results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
    trained_models[name] = model
    
    print(f"{name:25s} | RMSE: {rmse:>12,.0f} | MAE: {mae:>12,.0f} | R²: {r2:.4f}")

# Tableau récapitulatif
print("\n--- Classement par R² ---")
results_df = pd.DataFrame(results).T.sort_values('R2', ascending=False)
print(results_df.round(4))

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300, 500],
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True, False]
}

random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=50,
    cv=5,
    scoring='r2',
    n_jobs=1,
    random_state=42,
    verbose=1
)

random_search.fit(X_train, y_train)
best_model = random_search.best_estimator_
print("Optimisation terminee")

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

random_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42),
    param_distributions=param_dist,
    n_iter=20,
    cv=3,
    scoring='r2',
    n_jobs=1,
    random_state=42,
    verbose=0
)

random_search.fit(X_train, y_train)
best_model = random_search.best_estimator_
print("Optimisation terminee")
print(random_search.best_params_)

In [ ]:
y_pred_best = best_model.predict(X_test)

rmse_best = np.sqrt(mean_squared_error(y_test, y_pred_best))
mae_best = mean_absolute_error(y_test, y_pred_best)
r2_best = r2_score(y_test, y_pred_best)

print("RMSE :", round(rmse_best, 2))
print("MAE  :", round(mae_best, 2))
print("R2   :", round(r2_best, 4))

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry
import pandas as pd

session = get_active_session()

reg = Registry(
    session=session,
    database_name="HOUSE_PRICE_DB",
    schema_name="ML"
)

# Supprimer si déjà existant
try:
    reg.delete_model("HOUSE_PRICE_PREDICTOR")
    print("Ancien modèle supprimé")
except:
    print("Aucun modèle existant")

input_df = pd.DataFrame(X_train, columns=feature_cols)

model_version = reg.log_model(
    model=best_model,
    model_name="HOUSE_PRICE_PREDICTOR",
    version_name="V1",
    comment="Random Forest - R2=0.9112",
    metrics={
        "RMSE": float(rmse_best),
        "MAE": float(mae_best),
        "R2": float(r2_best)
    },
    sample_input_data=input_df
)

print("Modele enregistre avec succes !")
print(reg.show_models())

In [ ]:
# Charger le modèle depuis le registry
loaded_model = reg.get_model("HOUSE_PRICE_PREDICTOR").version("V1")

# Tester sur quelques nouvelles maisons
new_houses = pd.DataFrame({
    'AREA':             [3000, 1500, 4500],
    'BEDROOMS':         [3,    2,    5],
    'BATHROOMS':        [2,    1,    3],
    'STORIES':          [2,    1,    3],
    'MAINROAD':         [1,    1,    1],
    'GUESTROOM':        [0,    0,    1],
    'BASEMENT':         [1,    0,    1],
    'HOTWATERHEATING':  [0,    0,    1],
    'AIRCONDITIONING':  [1,    0,    1],
    'PARKING':          [2,    1,    3],
    'PREFAREA':         [1,    0,    1],
    'FURNISHINGSTATUS': [2,    0,    2]
})

# Normaliser avec le même scaler
new_houses_scaled = scaler.transform(new_houses)
new_input_df = pd.DataFrame(new_houses_scaled, columns=feature_cols)

# Prédire
predictions = loaded_model.run(new_input_df, function_name="predict")
print("Predictions :")
print(predictions)